In [ ]:
# 실습 준비 — 13주차 회귀분석
# 이 셀을 먼저 한 번 실행하세요. 데이터가 없으면 아래 셀들이 전부 실패합니다.
import os, pathlib, urllib.request

BASE = "https://raw.githubusercontent.com/aprilslab/statistics-lab/main/data/"
FILES = ["ch12_scores_reg.csv"]

pathlib.Path("data").mkdir(exist_ok=True)
for name in FILES:
    for dest in (pathlib.Path(name), pathlib.Path("data") / name):
        if not dest.exists():
            urllib.request.urlretrieve(BASE + name, dest)

# '../data/x.csv' 로 읽는 노트북 대응 — 상위 폴더에도 같은 data/ 를 걸어둔다.
# 절대경로(/data)로 박으면 cwd 가 /content 가 아닐 때 깨지므로 상대경로로 건다.
try:
    parent = pathlib.Path("..") / "data"
    if not parent.exists():
        os.symlink(pathlib.Path("data").resolve(), parent)
except OSError:
    pass

print("준비 완료:", ", ".join(FILES) if FILES else "(내려받을 데이터 없음)")


# 13주차 · 회귀분석 — 실습

수업 슬라이드의 `[셀 N]` 코드를 같은 순서로 담았습니다. 맨 위부터 차례로 실행하며 슬라이드의 실행 결과와 같은지 확인하세요.

## Colab 따라하기 ① — 이 표의 재료, 학생 20명  (슬라이드 3장)

홈의 Colab 칩으로 실습 노트북을 열고, 셀을 위에서부터 차례로 실행한다. 표의 재료는 교재 12장의 학생 20명이다.

In [ ]:
# [셀 1] 준비 — 그대로 실행
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'   # 결과를 줄마다 모두 보인다

In [ ]:
# [셀 2] 오늘 쓸 데이터
df = pd.read_csv('ch12_scores_reg.csv')
df.head(3)    # 앞 세 행 — 한 행이 학생 한 명

## Colab 따라하기 ② — 함께 움직이는 세기  (슬라이드 4장)

상관계수는 두 변수가 함께 움직이는 정도(세기)만 말한다. 회귀분석이란 원인이 되는 변수(설명변수)로 결과가 되는 변수(반응변수)를 설명 · 예측하는 식을 세우는 분석이다.

In [ ]:
# [셀 3] 상관계수
df[['final_test', 'quiz', 'sleep_time']].corr()

## Colab 따라하기 ③ — 한 줄로 이 표가 나온다  (슬라이드 5장)

공식은 '반응변수 ~ 설명변수' 로 쓴다. fit() 이 계산하고, summary() 가 오늘 읽을 표를 보인다.

In [ ]:
# [셀 4] 단순회귀 — 기말 점수 ~ 쪽지시험
result = smf.ols('final_test ~ quiz', df).fit()
result.summary()

## Colab 따라하기 ④ — 표의 칸을 이름으로 꺼내기  (슬라이드 17장)

보고서에 옮길 값은 이름으로 꺼낼 수 있다. 오른쪽 표의 quiz 행과 같은 숫자가 나온다.

In [ ]:
# [셀 5] 계수 · p값 · 95% 신뢰구간
result.params
result.pvalues
result.conf_int()

## Colab 따라하기 ⑤ — 설명변수 둘: 모형 2 의 표  (슬라이드 27장)

중회귀모형이란 설명변수가 둘 이상인 회귀모형이다. 같은 모양의 표에 행이 하나 늘었다. 계수는 '다른 설명변수가 같을 때' 그 변수가 1 클 때의 차이다.

In [ ]:
# [셀 6] 모형 2 — 쪽지시험 + 수면 시간
m2 = smf.ols('final_test ~ quiz + sleep_time', df).fit()
m2.summary()

## Colab 따라하기 ⑥ — 글자로 된 변수: 통학 방법의 표  (슬라이드 28장)

가변수란 질적변수를 0 과 1 로 바꾼 변수다. 범주가 3 개면 가변수는 2 개 — 표에 행 이름이 없는 범주(bicycle)가 비교의 기준 범주다.

In [ ]:
# [셀 7] 통학 방법만
smf.ols('final_test ~ school_method', df).fit().summary()

## Colab 따라하기 ⑦ — 다른 변수를 더하면: 모형 3 의 표  (슬라이드 29장)

쪽지시험 · 수면 시간을 함께 넣으면 walk 계수가 −19.00 에서 −7.66 (p = 0.252)으로 줄고 유의하지 않다. 자전거로 오는 학생들의 쪽지시험 평균이 원래 높았기 때문이다.

In [ ]:
# [셀 8] 모형 3 — 쪽지시험 + 수면 시간 + 통학 방법
m3 = smf.ols('final_test ~ quiz + sleep_time + school_method', df).fit()
m3.summary()

## Colab 따라하기 ⑧ — 세 표의 두 칸을 한 표로  (슬라이드 33장)

세 표를 번갈아 보는 대신, rsquared_adj 와 aic 를 꺼내 한 표로 모은다. 보고서 3-4 의 비교표가 이것이다.

In [ ]:
# [셀 9] 세 모형의 비교표
m1 = smf.ols('final_test ~ quiz', df).fit()
pd.DataFrame({'조정결정계수': [m1.rsquared_adj, m2.rsquared_adj, m3.rsquared_adj],
              'AIC': [m1.aic, m2.aic, m3.aic]}, index=['모형 1', '모형 2', '모형 3'])

## Colab 따라하기 ⑨ — 모형 2 의 잔차 그림  (슬라이드 38장)

고른 모형 2 의 잔차를 예측값에 대해 점으로 찍고 0 선을 긋는다. Colab 에는 한글 글꼴이 없어 축 이름을 영어로 쓴다.

In [ ]:
# [셀 10] 잔차 그림 — 줄 끝 ; 은 그림 말고 다른 글자가 찍히지 않게 한다
plt.scatter(m2.fittedvalues, m2.resid);   # 가로 예측값 · 세로 잔차
plt.axhline(0, color='grey');             # 0 선
plt.xlabel('fitted');
plt.ylabel('residual');